In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

In [ ]:
def load_vds_files(folder_path):
    vds_data = []
    for file_path in glob.glob(os.path.join(folder_path, "vds_*.xlsx")):
        year = int(file_path.split("_")[-1].split(".")[0])
        df = pd.read_excel(file_path)
        df_melted = df.melt(id_vars=["Отрасль"], var_name="Region", value_name="VDS_Value")
        df_melted["Year"] = year
        vds_data.append(df_melted)
    return pd.concat(vds_data, ignore_index=True)

vds_folder_path = "./data"
vds_df = load_vds_files(vds_folder_path)

In [ ]:
vds_df

In [ ]:
sphere_mapping = {
    "Снабжение электроэнергией, газом, паром, горячей водой и кондиционированным воздухом": "Электроснабжение, подача газа, пара и воздушное кондиционирование",
    "Водоснабжение сбор, обработка и удаление отходов; деятельность поликвидации загрязнений": "Водоснабжение; канализационная система, контроль над сбором и распределением отходов",
    "Здравоохранение и социальное обслуживание населения": "Здравоохранение и социальные услуги"
}
vds_df["Отрасль"] = vds_df["Отрасль"].replace(sphere_mapping)

In [ ]:
city_mapping = {
    "г. Нур-Султан": "г. Астана",
    "Карагандиская": "Карагандинская"
}
vds_df["Region"] = vds_df["Region"].replace(city_mapping)

In [ ]:
vds_df['Отрасль'].unique()

In [ ]:
vds_df['Region'].unique()

In [ ]:
def load_gdp_file(gdp_file_path):
    # Load GDP data
    gdp_df = pd.read_excel(gdp_file_path)
    gdp_df_long = gdp_df.melt(id_vars=["Год"], var_name="Region", value_name="GDP")
    gdp_df_long = gdp_df_long.rename(columns={"Год": "Year"})
    
    return gdp_df_long

# Specify the path to your GDP file
gdp_file_path = "./data/gdp.xlsx"
gdp_df = load_gdp_file(gdp_file_path)

In [ ]:
gdp_df = gdp_df.dropna()

In [ ]:
gdp_df["Region"] = gdp_df["Region"].replace(city_mapping)

In [ ]:
gdp_df["Region"].unique()

In [ ]:
gdp_df

In [ ]:
def create_lagged_features(df, lag_years=1):
    df = df.sort_values(by=["Region", "Year"]).reset_index(drop=True)

    # lagged features (gdp)
    for lag in range(1, lag_years + 1):
        df[f"GDP_Lag{lag}"] = df.groupby("Region")["GDP"].shift(lag)
        
    
    return df

# Create lagged features with 1 year lag for GDP, add VDS lag if desired
gdp_df = create_lagged_features(gdp_df, lag_years=1)

In [ ]:
gdp_df

In [ ]:
merged_df = pd.merge(vds_df, gdp_df, on=["Year", "Region"], how="left")

In [ ]:
merged_df

In [ ]:
final_df = merged_df

In [ ]:
final_df

In [ ]:
final_df['Region'].unique()

In [ ]:
final_df = pd.get_dummies(final_df, columns=["Region"], drop_first=True)

In [ ]:
final_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
final_df = merged_df
final_df = pd.get_dummies(final_df, columns=["Region"], drop_first=True)

vds_totals = final_df.groupby("Year")["VDS_Value"].sum().reset_index()
final_df = final_df.merge(vds_totals, on="Year", suffixes=("", "_YearTotal"))

for region_col in [col for col in final_df.columns if col.startswith("Region_")]:
    region_data = final_df[final_df[region_col] == 1]
    region_data["Sector_Contribution"] = region_data["VDS_Value"] / region_data["VDS_Value_YearTotal"] * 100
    top_sectors = region_data.groupby("Отрасль")["Sector_Contribution"].mean().nlargest(3).index

    plt.figure(figsize=(10, 6))
    for sector in top_sectors:
        sector_data = region_data[region_data["Отрасль"] == sector]
        plt.plot(sector_data["Year"], sector_data["Sector_Contribution"], label=sector)
    plt.title(f"Top 3 Sector Contributions Over Time in {region_col.replace('Region_', '')}")
    plt.xlabel("Year")
    plt.ylabel("Sector Contribution (%)")
    plt.legend(title="Sector")
    plt.show()

In [ ]:
final_df = merged_df
final_df = pd.get_dummies(final_df, columns=["Region"], drop_first=True)

vds_totals = final_df.groupby("Year")["VDS_Value"].sum().reset_index()
final_df = final_df.merge(vds_totals, on="Year", suffixes=("", "_YearTotal"))
region_sector_rankings = {}

for region_col in [col for col in final_df.columns if col.startswith("Region_")]:
    region_data = final_df[final_df[region_col] == 1]
    region_data["Sector_Contribution"] = region_data["VDS_Value"] / region_data["VDS_Value_YearTotal"] * 100
    avg_contributions = region_data.groupby("Отрасль")["Sector_Contribution"].mean().sort_values(ascending=False)
    region_name = region_col.replace("Region_", "")
    region_sector_rankings[region_name] = avg_contributions
rankings_df = pd.DataFrame(region_sector_rankings)


In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(rankings_df, annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={'label': 'Average Contribution (%)'})
plt.title("Sector Contribution Rankings by Region")
plt.xlabel("Region")
plt.ylabel("Sector")
plt.show()

In [ ]:
normalized_rankings_df = rankings_df.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=0)

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(normalized_rankings_df, annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={'label': 'Normalized Contribution (0-1)'})
plt.title("Normalized Sector Contribution Rankings by Region")
plt.xlabel("Region")
plt.ylabel("Sector")
plt.show()